In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

: 

In [ ]:
df_full = pd.read_csv('../data/raw/student_mental_health_burnout_1M.csv')

# Balanced 15K per class
n_per_class = 15000
df = df_full.groupby('risk_level', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), n_per_class), random_state=42)
).reset_index(drop=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sample shape : {df.shape}")
print(f"\nDistribution:")
for lvl in ['Low', 'Medium', 'High']:
    cnt = (df['risk_level'] == lvl).sum()
    print(f"  {lvl:<10}: {cnt:,}")
df.head()

Sample shape : (45000, 20)

Distribution:
  Low       : 15,000
  Medium    : 15,000
  High      : 15,000


C:\Users\zahra\AppData\Local\Temp\ipykernel_29572\4057086898.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_full.groupby('risk_level', group_keys=False).apply(


,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,anxiety_score,depression_score,sleep_hours,physical_activity,social_support,screen_time,internet_usage,financial_stress,family_expectation,burnout_score,mental_health_index,risk_level,dropout_risk
0,20,Female,4,2.730820,4.850862,60.300038,5.243260,4.107601,1.685632,6.730780,3.792768,4.803484,7.168959,7.153069,7.875235,5.087090,3.080292,6.164726,Medium,2.810641
1,21,Male,4,6.006462,6.989138,72.498312,7.912069,5.178125,4.469066,4.443861,0.970499,4.663864,5.116087,5.547409,4.790890,8.260161,7.554783,3.941015,High,4.964135
2,19,Female,2,8.647586,7.997322,69.182170,4.804022,5.524058,2.782752,5.273105,5.069558,7.784344,1.466554,1.073356,4.444326,6.325128,3.085731,5.586348,Medium,1.629859
3,28,Male,1,5.575063,6.949871,69.122998,6.487279,4.156164,3.775590,3.938624,6.225925,5.923078,9.966368,9.536901,3.938840,8.636600,6.290635,5.025562,High,3.847302
4,23,Female,3,7.558773,7.945329,68.566738,7.083566,4.843014,4.744509,3.307270,1.775822,2.853231,6.512696,4.568547,7.095896,4.570003,5.031301,4.290316,Medium,5.552106


In [ ]:
# Encode target
risk_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['risk_encoded'] = df['risk_level'].map(risk_mapping)

# Encode gender
gender_mapping = {'Male': 0, 'Female': 1, 'Other': 2}
df['gender_encoded'] = df['gender'].map(gender_mapping)

print("Risk Level Encoding:")
for k, v in risk_mapping.items():
    print(f"{k} to {v}")

print("\nGender Encoding:")
for k, v in gender_mapping.items():
    print(f"{k} to {v}")

print(f"\nSample:")
print(df[['risk_level', 'risk_encoded', 'gender', 'gender_encoded']].drop_duplicates().sort_values('risk_encoded').to_string(index=False))

Risk Level Encoding:
Low to 0
Medium to 1
High to 2

Gender Encoding:
Male to 0
Female to 1
Other to 2

Sample:
risk_level  risk_encoded gender  gender_encoded
       Low             0   Male               0
       Low             0 Female               1
       Low             0  Other               2
    Medium             1 Female               1
    Medium             1   Male               0
    Medium             1  Other               2
      High             2   Male               0
      High             2 Female               1
      High             2  Other               2


In [ ]:
FEATURES = ['age', 'gender_encoded', 'academic_year',
            'study_hours_per_day', 'exam_pressure', 'academic_performance',
            'stress_level', 'anxiety_score', 'depression_score',
            'sleep_hours', 'physical_activity', 'social_support',
            'screen_time', 'internet_usage', 'financial_stress',
            'family_expectation', 'burnout_score', 'mental_health_index',
            'dropout_risk']

X = df[FEATURES]
y = df['risk_encoded']

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"\nFeatures ({len(FEATURES)}):")
for f in FEATURES:
    print(f"- {f}")
print(f"\nClass distribution:")
print(y.value_counts().sort_index().rename({0:'Low', 1:'Medium', 2:'High'}))

X shape : (45000, 19)
y shape : (45000,)

Features (19):
- age
- gender_encoded
- academic_year
- study_hours_per_day
- exam_pressure
- academic_performance
- stress_level
- anxiety_score
- depression_score
- sleep_hours
- physical_activity
- social_support
- screen_time
- internet_usage
- financial_stress
- family_expectation
- burnout_score
- mental_health_index
- dropout_risk

Class distribution:
risk_encoded
Low       15000
Medium    15000
High      15000
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nTrain class distribution:")
print(y_train.value_counts().sort_index().rename({0:'Low', 1:'Medium', 2:'High'}))
print(f"\nTest class distribution:")
print(y_test.value_counts().sort_index().rename({0:'Low', 1:'Medium', 2:'High'}))

X_train : (36000, 19)
X_test  : (9000, 19)

Train class distribution:
risk_encoded
Low       12000
Medium    12000
High      12000
Name: count, dtype: int64

Test class distribution:
risk_encoded
Low       3000
Medium    3000
High      3000
Name: count, dtype: int64


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURES, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=FEATURES, index=X_test.index)

print("Scaling done ✓")
print(f"\nX_train mean:")
print(X_train_scaled.mean().round(3))
print(f"\nX_train std:")
print(X_train_scaled.std().round(3))